In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [3]:
import torch
import torch.nn as nn   
import torch.optim as optim

from torch.optim import SGD

from torch.utils.data import dataloader, TensorDataset

Input and target data:

In [15]:
input_flux = np.load("../preprocessing/preprocess_cflib_vyom.npy")

target_flux = np.load("interpolated_flux_data.npz")["flux"]

print(len(target_flux[0]))  # checking the length of the interpolated flux arrays
print(input_flux)
print(target_flux)


4306
[[1.18421388e+00 1.18500031e+00 1.18583710e+00 ... 4.12755520e-01
  4.12429333e-01 4.12375290e-01]
 [5.08645436e-02 5.23604861e-02 5.36947523e-02 ... 1.50170778e+00
  1.49373260e+00 1.48894048e+00]
 [2.34775780e-01 2.39047940e-01 2.42083210e-01 ... 8.05570383e-01
  8.01278850e-01 8.00003801e-01]
 ...
 [6.50488264e-01 6.54929602e-01 6.58272352e-01 ... 9.86451435e-05
  9.86451435e-05 9.86451435e-05]
 [4.07824568e-01 4.11455545e-01 4.13382647e-01 ... 9.34917567e-05
  9.34917567e-05 9.34917567e-05]
 [6.21511827e-01 6.25559637e-01 6.29292880e-01 ... 6.39930180e-01
  6.39071734e-01 6.38985885e-01]]
[[1.2201196  1.2343417  1.2461157  ... 0.42294842 0.42489082 0.42346513]
 [0.02008478 0.02623232 0.03615447 ... 1.5132846  1.5031068  1.5021192 ]
 [0.13137466 0.14902087 0.15827061 ... 0.8248837  0.8144983  0.80260134]
 ...
 [0.7116064  0.75607896 0.7684375  ... 0.6480675  0.64545834 0.6394347 ]
 [0.25271845 0.33104327 0.34298497 ... 0.7123951  0.7035924  0.6973554 ]
 [0.7862435  0.82383054 0

Initializing the Denoising Auto Encoder :

In [ ]:
class DenoisingAutoencoders(nn.modules):
    def __init__(self, input_dim = 4306, latent_dim = 128):
        super(DenoisingAutoencoders, self).__init__()

        # initializing the latent space (bottle neck for the network)

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024,512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256,latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256,512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024,2048),
            nn.ReLU(),
            nn.Linear(2048,4306)
        )

    # x here is a datapoint --> in our case a rank one tensor (datapoint vector)
    
    def forward(self, x):       
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        return decoded
    
    # class method

    @classmethod
    def add_noise(x, noise_factor = 0.01):      # add a gaussian noise to the data vector 
        noise = noise_factor * torch.randn_like(x)       # choose a random tensor with a same shape as X from standard normal distribution
        return x + noise
    

